# Проект по итогу Спринта 6. Предобработка и анализ данных по играм для различных платформ для подготовки аналитической статьи

- Автор: Шмарук Александр
- Дата: 09.11.2025

### Цели и задачи проекта

<font color='#777778'>
    <b>Цель проекта:</b> провести анализ данных о продажах, рейтингах игр и популярности игровых платформ в начала XXI века на основе предоставленной выгрузки данных
    <br><b>Задачи проекта:</b> подготовить данные, провести выборку и сделать анализ согласно заданным критериям
</font>

### Описание данных

<font color='#777778'>Данные представляют собой CSV файл, содержащий статистику по продажам игр для различных платформ в разных регионах мира, а также данные по рейтингам пользователей и критиков</font>

### Содержимое проекта

<font color='#777778'>
1. Загрузка и предварительное изучение набора данных 
<br>2. Изучение типов данных и их анализ
<br>3. Обработка данных: работа с пропусками и дубликатами
<br>4. Подготовка фильтра данных по заданию
<br>5. Категоризация данных
<br>6. Подготовка и описание выводов по задаче</font>

---

## 1. Загрузка данных и знакомство с ними

In [1]:
# Пустые ячейки после каждого задания — примерное пространство для работы.
# Вы можете свободно добавлять или удалять ячейки по своему усмотрению в зависимости от логики и объёма работы.

In [2]:
# Используйте ячейки типа Code для вашего кода

In [3]:
# При необходимости добавьте новые ячейки для кода

In [4]:
import pandas as pd

In [5]:
df = pd.read_csv('https://code.s3.yandex.net/datasets/new_games.csv')

# Отключим Warn для работы с срезами от исходного дата фрейма для красоты
pd.options.mode.chained_assignment = None 

# Сохраним число строк в исходном дата фрейме

initial_df_rows = df.shape[0]

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


In [7]:
# Выведем первые 5 строк для визуального просмотра данных в строках

df.head()

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


Объем данных: 16956 строк, 11 столбцов. Объем данных в целом соответствует задаче, но в части столбцов (6 шт) есть пропуски. 
Названия столбцов - соответствут содержимому, кроме столбца Rating. Его будет необходимо переименовать. Также в некоторых столбцах есть данные не верного - Год выпуска не приведен к целочисленному типу, рейтинг пользовательский с типом Object - потребуется привести данные к нужным типам для их корректного анализа 

---

## 2.  Проверка ошибок в данных и их предобработка

### 2.1. Названия, или метки, столбцов датафрейма

In [8]:
df.columns

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')

In [9]:
df = df.rename(columns = {'Name': 'name',
                          'Platform': 'platform',
                          'Year of Release': 'year_of_release',
                          'Genre': 'genre',
                          'NA sales': 'na_sales',
                          'EU sales': 'eu_sales',
                          'JP sales': 'jp_sales',
                          'Other sales': 'other_sales',
                          'Critic Score': 'critic_score',
                          'User Score': 'user_score',
                          'Rating': 'esrb_rating'})              

In [10]:
df.columns

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'esrb_rating'],
      dtype='object')

### 2.2. Типы данных

In [11]:
df.dtypes

name                object
platform            object
year_of_release    float64
genre               object
na_sales           float64
eu_sales            object
jp_sales            object
other_sales        float64
critic_score       float64
user_score          object
esrb_rating         object
dtype: object

Необходимо преобразовать тип данных для столбцов: year_of_release в int, user_score в float, eu_sales и jp_sales в float
Также необходимо обработать пропуски: в столбцах name, year_of_release, genre пропуски можно удалить - их % незначителен к объему данных. 
В столбцах  user_score и critic_score пропущенных данных 40-50%, потребуется их обработать

In [12]:
# Преобразуем тип данных в нужных столбцах. Для столбца year_of_release выделим год

df['year_of_release'] = pd.to_datetime(df['year_of_release'], format = '%Y', errors = 'coerce') 
df['year_of_release'] = df['year_of_release'].dt.year
df['eu_sales'] = pd.to_numeric(df['eu_sales'], errors = 'coerce', downcast = 'float')
df['jp_sales'] = pd.to_numeric(df['jp_sales'], errors = 'coerce', downcast = 'float') 
df['user_score'] = pd.to_numeric(df['user_score'], errors = 'coerce', downcast = 'float') 

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16956 non-null  object 
 2   year_of_release  16681 non-null  float64
 3   genre            16954 non-null  object 
 4   na_sales         16956 non-null  float64
 5   eu_sales         16950 non-null  float32
 6   jp_sales         16952 non-null  float32
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       7688 non-null   float32
 10  esrb_rating      10085 non-null  object 
dtypes: float32(3), float64(4), object(4)
memory usage: 1.2+ MB


### 2.3. Наличие пропусков в данных

In [14]:
#### Количество пропусков в столбцах
df.isna().sum()

name                  2
platform              0
year_of_release     275
genre                 2
na_sales              0
eu_sales              6
jp_sales              4
other_sales           0
critic_score       8714
user_score         9268
esrb_rating        6871
dtype: int64

In [15]:
#### Относительная доля пропусков в столбцах в %
round(df.isna().mean() * 100, 2)

name                0.01
platform            0.00
year_of_release     1.62
genre               0.01
na_sales            0.00
eu_sales            0.04
jp_sales            0.02
other_sales         0.00
critic_score       51.39
user_score         54.66
esrb_rating        40.52
dtype: float64

Пропуски в столбцах: name, year_of_release, genre, eu_sales, jp_sales не значительны, могли возникнуть из-за некорректного сбора или слияния данных из различных источников. Пропуски в названиях, жанре и годе выпуска можно удалить (пренебречь), пропуски в данных по продажам можно заменить на средние значения в зависимости от названия платформы и года выхода игры

Пропусков в critic_score и user_score достаточно большое количество, может быть связано с тем, что не по всем продажам были заполнены отзывы, либо не во всех странах обе этих оценки запрашивались во все годы продаж
Пропуски в esrb_rating возможно, связаны с тем, что не во всех странах необходима маркировка во все годы сбора данных или пустое значение может соответствовать рейтингу Для всех возрастов

In [16]:
# Удаляем пропуски в столбцах, которые не повлияют на исследование
df = df.dropna(subset = ['name', 'year_of_release', 'genre'])

In [17]:
# Заменим пропуски в данных о продажах на средние по платформе и году

# Напишем функцию для определения среднего значения eu_sales по необходимым параметрам, с учетом, что заменять нужно только пропущенные значения
def eu_sales_missings(row):
    if pd.isna(row['eu_sales']):
        group = df[(df['platform'] == row['platform']) & (df['year_of_release'] == row['year_of_release'])]
        return group['eu_sales'].mean()
    else: 
        return row['eu_sales']

# Применим функцию для заполнения пропусков в eu_sales

df['eu_sales'] = df.apply(eu_sales_missings, axis=1)

# Напишем функцию для определения среднего значения jp_sales по необходимым параметрам, с учетом, что заменять нужно только пропущенные значения
def jp_sales_missings(row):
    if pd.isna(row['jp_sales']):
        group = df[(df['platform'] == row['platform']) & (df['year_of_release'] == row['year_of_release'])]
        return group['jp_sales'].mean()
    else: 
        return row['jp_sales']

# Применим функцию для заполнения пропусков в jp_sales

df['jp_sales'] = df.apply(eu_sales_missings, axis=1)

In [18]:
# Запросим инфо по датафрейму, для подтверждения, что пропусков в обработанных столбцах нет

df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16679 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16679 non-null  object 
 1   platform         16679 non-null  object 
 2   year_of_release  16679 non-null  float64
 3   genre            16679 non-null  object 
 4   na_sales         16679 non-null  float64
 5   eu_sales         16679 non-null  float64
 6   jp_sales         16679 non-null  float64
 7   other_sales      16679 non-null  float64
 8   critic_score     8085 non-null   float64
 9   user_score       7558 non-null   float32
 10  esrb_rating      9901 non-null   object 
dtypes: float32(1), float64(6), object(4)
memory usage: 1.5+ MB


### 2.4. Явные и неявные дубликаты в данных

In [19]:
# Нормализуем данные

df['name'] = df['name'].str.lower()
df['genre'] = df['genre'].str.lower()
df['platform'] = df['platform'].str.upper()

# Выведем список уникальных жанров

uniqie_genre = df['genre'].unique()
print('Genres list: ', uniqie_genre)

# Выведем список уникальных платформ

uniqie_platform = df['platform'].unique()
print('Platforms list: ', uniqie_platform)

# Выведем список годов выпуска

unique_year_of_release = df['year_of_release'].unique()
print('Years list: ', unique_year_of_release)

# Выведем список значений рейтинга ESRB

unique_esrb_rating = df['esrb_rating'].unique()
print('ESRB list: ', unique_esrb_rating)

Genres list:  ['sports' 'platform' 'racing' 'role-playing' 'puzzle' 'misc' 'shooter'
 'simulation' 'action' 'fighting' 'adventure' 'strategy']
Platforms list:  ['WII' 'NES' 'GB' 'DS' 'X360' 'PS3' 'PS2' 'SNES' 'GBA' 'PS4' '3DS' 'N64'
 'PS' 'XB' 'PC' '2600' 'PSP' 'XONE' 'WIIU' 'GC' 'GEN' 'DC' 'PSV' 'SAT'
 'SCD' 'WS' 'NG' 'TG16' '3DO' 'GG' 'PCFX']
Years list:  [2006. 1985. 2008. 2009. 1996. 1989. 1984. 2005. 1999. 2007. 2010. 2013.
 2004. 1990. 1988. 2002. 2001. 2011. 1998. 2015. 2012. 2014. 1992. 1997.
 1993. 1994. 1982. 2016. 2003. 1986. 2000. 1995. 1991. 1981. 1987. 1980.
 1983.]
ESRB list:  ['E' nan 'M' 'T' 'E10+' 'K-A' 'AO' 'EC' 'RP']


In [20]:
# Определим количество явных дубликатов

df.duplicated(subset = ['name', 'genre', 'platform', 'year_of_release']).sum() 

236

In [21]:
# Очистим dataframe от явных дубликатов по столбцам Жанр, Платформа, Год, Название игры

df_cleaned = df.drop_duplicates(subset = ['name', 'genre', 'platform', 'year_of_release'])

In [22]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16443 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16443 non-null  object 
 1   platform         16443 non-null  object 
 2   year_of_release  16443 non-null  float64
 3   genre            16443 non-null  object 
 4   na_sales         16443 non-null  float64
 5   eu_sales         16443 non-null  float64
 6   jp_sales         16443 non-null  float64
 7   other_sales      16443 non-null  float64
 8   critic_score     7982 non-null   float64
 9   user_score       7462 non-null   float32
 10  esrb_rating      9767 non-null   object 
dtypes: float32(1), float64(6), object(4)
memory usage: 1.4+ MB


Данные были приведены в единнобразный вид.
Далее были выделены дублирующиеся записи по Названию игры, году релиза, платформе, жанру и удалены

In [23]:
# Посчитаем число удаленных дубликатов - сравним 2 датафрейма

deleted_duplicated_rows = df.shape[0] - df_cleaned.shape[0]
print('Deleted: ',deleted_duplicated_rows, ' rows')

Deleted:  236  rows


In [24]:
# Посчитаем относительное и абсолютное число удаленных строк

abs_deleted_missings = initial_df_rows - df.shape[0]
relative_deleted_missings = round((initial_df_rows - df.shape[0]) / initial_df_rows, 2)

print(f'After missings cleaning it was deleted {abs_deleted_missings} rows or {relative_deleted_missings} share')

abs_deleted = initial_df_rows - df_cleaned.shape[0]
relative_deleted = round((initial_df_rows - df_cleaned.shape[0]) / initial_df_rows, 2)

print(f'After missings and duplicates cleaning it was deleted {abs_deleted} rows or {relative_deleted} share')

After missings cleaning it was deleted 277 rows or 0.02 share
After missings and duplicates cleaning it was deleted 513 rows or 0.03 share


В целом, можно сделать вывод что количество данных, которые были удалены в рамках подготовки к анализу - достаточно небольшое, менее 5%.
Далее можно использовать данные для фильтрации и анализа. 

---

## 3. Фильтрация данных

In [25]:
# Отберем необходимые данные по условию Год выхода игры от 2000 до 2013 включительно

df_actual = df_cleaned[(df_cleaned['year_of_release'] >= 2000) & (df_cleaned['year_of_release'] <= 2013)]

In [26]:
# Проверим информацию о полученном срезе

df_actual.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 12780 entries, 0 to 16954
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             12780 non-null  object 
 1   platform         12780 non-null  object 
 2   year_of_release  12780 non-null  float64
 3   genre            12780 non-null  object 
 4   na_sales         12780 non-null  float64
 5   eu_sales         12780 non-null  float64
 6   jp_sales         12780 non-null  float64
 7   other_sales      12780 non-null  float64
 8   critic_score     7168 non-null   float64
 9   user_score       6482 non-null   float32
 10  esrb_rating      8722 non-null   object 
dtypes: float32(1), float64(6), object(4)
memory usage: 1.1+ MB


---

## 4. Категоризация данных

In [27]:
# Подготовим функцию категоризации по оценкам пользователей

def user_categorize(row):
    if (row['user_score'] >= 8) & (row['user_score'] <= 10):
        return "Высокая оценка"
    elif (row['user_score'] < 8) & (row['user_score'] >= 3):
        return "Средняя оценка"
    elif (row['user_score'] < 3) & (row['user_score'] >= 0):
        return "Низкая оценка"
    else: 
        pass

# Применим функцию для дата фрейма и создадим новый столбец User_score_category

df_actual['user_score_category'] = df_actual.apply(user_categorize, axis=1)

In [28]:
def critic_categorize(row):
    if (row['critic_score'] >= 80) & (row['critic_score'] <= 100):
        return "Высокая оценка"
    elif (row['critic_score'] < 80) & (row['critic_score'] >= 30):
        return "Средняя оценка"
    elif (row['critic_score'] < 30) & (row['critic_score'] >= 0):
        return "Низкая оценка"
    else: 
        pass

# Применим функцию для дата фрейма и создадим новый столбец Critic_score_category

df_actual['critic_score_category'] = df_actual.apply(critic_categorize, axis=1)

# - После категоризации данных проверьте результат: сгруппируйте данные по выделенным категориям и посчитайте количество игр в каждой категории.

In [29]:
# Посчитаем число оценок по категориям Оценка пользователей

grouped_user_df = df_actual.groupby('user_score_category')['name'].count()
print(grouped_user_df)

user_score_category
Высокая оценка    2286
Низкая оценка      116
Средняя оценка    4080
Name: name, dtype: int64


Вывод: пользовательская оценка присутствует примерно в половине игр. Низких оценок достаточно мало, это может свидетельствовать о том, что для части игр пользователи не ставят оценку, а просто переходят к другим

In [30]:
# Посчитаем число оценок по категориям Оценка критиков

grouped_critic_df = df_actual.groupby('critic_score_category')['name'].count()
print(grouped_critic_df)

critic_score_category
Высокая оценка    1691
Низкая оценка       55
Средняя оценка    5422
Name: name, dtype: int64


Вывод: оценок критиков больше, чем оценок пользователей, однако достаточно большой пласт игр без оценок, что может говорить о том, что большая часть игр не пользовалась особым спросом и оценок критиков не запрашивалось. Основная оценка - Средняя. Можно предположить, что слишком большой промежуток задан для Средней категории, возможно, стоит еще разбить данные на группы

In [31]:
# Сгруппируем игры по платформам за 2000-2013 годы, отсортируем по убыванию количества игр для платформы
platform_group = df_actual.groupby('platform')['name'].count().sort_values(ascending = False)

# Выведем топ-7 платформ
print(platform_group[0:7])

platform
PS2     2127
DS      2120
WII     1275
PSP     1180
X360    1121
PS3     1086
GBA      811
Name: name, dtype: int64


Вывод: основная часть игр была выпущена для PS2 и DS  - что говорит о примерно равной конкуренции между основными производителями консолей тех лет

---

## 5. Итоговый вывод

В итоговый срез попали данные, у которых были обработаны пропуски и дубликаты, применена фильтрация по годам из задачи (2000 - 2013)
Вывод, который можно сделать сравнивая категоризацию Пользователей и Критиков: уровень оценки пользователей значительно выше, чем уровень оценки критиков. Однако, низкую оценку у пользователей получило большее число игр, что позволяет говорить о том, что пользователи оценивают более эмоционально, чем профессиональные критики.
В качестве дополнительных были добавлены столбцы: категория оценки от пользователей и категория оценки от критиков.
Оценка в актуальном срезе была проставлена примерно 50-55% игр, что повлияло в целом на итоговый результат - большой пласт данных не имеет оценки пользователей и критиков, и заполнить эти данные автоматически не возможно.

В период 2000-2013 годы основными игроками на рынке платформ были Sony и Nintendo. Далее с большим отставанием X360 от Microsoft